# Aula 24 - Notebook: Árvore de Atingibilidade e Árvore de Cobertura (Karp-Miller) em Redes de Petri
## Aplicação na Planta de Classificação Óptica de Grãos — SCADA-Core Automática (GRUPO 4)

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade Fabril:** Linha Automatizada de Seleção, Pesagem Dinâmica e Classificação Óptica de Grãos com Separação Pneumática Tripla (Arroz)  
**Disciplina:** Engenharia de Automação e Controle — ECAA08  

---

Este notebook implementa os algoritmos formais de **Árvore de Atingibilidade (*Reachability Tree*)** e **Árvore de Cobertura (*Coverability Tree* — Algoritmo de Karp-Miller)** aplicados ao controle a eventos discretos da nossa planta de seleção de grãos:

* **Recepção e Suprimento:** Funil de Recepção (`FN-101`, sensor `LIT-101`) e Central Pneumática (`CP-601`, transmissor `PT-601` / pressostato `PAL-601`).
* **Linha de Classificação:** Alimentador Vibratório (`ALIM-101`), Esteira de Alta Velocidade (`EST-201`), Balança Contínua (`BAL-301`), Túnel de Visão (`VIS-401`) e Ejetores (`FY-602`/`FY-603`).
* **Armazenamento e Logística:** Silo Coletor de Grãos Nobres (`SILO-701`, sensor `LIT-701`, comporta `XV-701`) e expedição autônoma pelo veículo `AGV-801` (Aula 22).

### Roteiro de Execução:
1. **Modelagem Matricial da Rede de Petri** ($N = (P, T, \text{Pre}, \text{Post}, M_0)$) via matriz de incidência $C = \text{Post} - \text{Pre}$.
2. **Cenário A (Rede Não-Limitada):** Construção da Árvore de Cobertura (Karp-Miller) detectando o símbolo $\omega$ (risco formal de *overflow* no `SILO-701` e no `FN-101`).
3. **Cenário B (Síntese de Controle por Lugar Complementar — Invariante-P):** Adição de capacidade física finita $K_{\max}$ ao `SILO-701` (`LIT-701`/`p_NC701`) e ao `FN-101` (`LIT-101`/`p_NC101`), acoplados à drenagem pelo `AGV-801`, gerando a **Árvore de Atingibilidade Finita**.
4. **Auditoria Automática de Propriedades Comportamentais:** Verificação de *Limitação (Boundedness)*, *Segurança (Safeness)*, *Ausência de Deadlocks*, *Reversibilidade*, *Vivacidade (L4-Live)* e *Invariantes-P*, além da plotagem do grafo de atingibilidade.

### 1. Definição Formal da Rede de Petri — Cenário A (Sem Limite Explícito no `SILO-701` e `FN-101`)

| Lugar | Descrição Física / Tag ISA-5.1 | Marcação Inicial $M_0$ |
| :--- | :--- | :---: |
| `P1_FN101_GRAOS` | Lote de grãos brutos disponível no Funil `FN-101` (`LIT-101` OK) | `1` |
| `P2_AR_COMP_OK` | Carga de ar comprimido pronta no Compressor `CP-601` (`¬p_PAL601`) | `1` |
| `P3_LINHA_LIVRE` | Linha `EST-201` + `VIS-401` ociosa e liberada (`c_PERM = 1`) | `1` |
| `P4_CLASSIFICANDO` | Lote em pesagem (`BAL-301`), visão (`VIS-401`) e ejeção (`FY-602/603`) | `0` |
| `P5_SILO701_LOTES` | Lotes de arroz aprovado acumulados no `SILO-701` (`LIT-701`) | `0` |

**Transições ($T$):**
* `T1_DOSAR_E_INSPEC`: Acionamento de `ALIM-101` (`c_ALIM`) e `EST-201` (`c_EST`) com pressão OK (consome 1 lote de `FN-101`, 1 unidade de ar de `CP-601` e ocupa a linha).
* `T2_EJETAR_SILO701`: Conclusão da classificação óptica/ejeção e descarga no `SILO-701` (libera a linha `P3_LINHA_LIVRE` e deposita 1 lote em `P5_SILO701_LOTES`).
* `T3_REABASTECER_REC`: Reposição contínua de 1 lote de grãos no funil `FN-101` e recarga pneumática do compressor `CP-601`.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque

class RedeDePetri:
    """Representação matricial de uma Rede de Petri Marcada N = (P, T, Pre, Post, M0)."""
    def __init__(self, lugares, transicoes, pre, post, m0):
        self.lugares = lugares
        self.transicoes = transicoes
        self.Pre = np.array(pre, dtype=int)   # Matriz |P| x |T| (arcos de entrada)
        self.Post = np.array(post, dtype=int) # Matriz |P| x |T| (arcos de saída)
        self.C = self.Post - self.Pre         # Matriz de Incidência C = Post - Pre
        self.M0 = tuple(m0)

    def exibir_matrizes(self, titulo=""):
        df_c = pd.DataFrame(self.C, index=self.lugares, columns=self.transicoes)
        sufixo = f" — {titulo}" if titulo else ""
        print(f"=== MATRIZ DE INCIDÊNCIA (C = Post - Pre){sufixo} ===")
        print(df_c.to_string())

# ==============================================================================
# CENÁRIO A: REDE SEM LIMITE FÍSICO EXPLÍCITO NO SILO-701 E NO FUNIL FN-101
# ==============================================================================
lugares_A = [
    "P1_FN101_GRAOS",
    "P2_AR_COMP_OK",
    "P3_LINHA_LIVRE",
    "P4_CLASSIFICANDO",
    "P5_SILO701_LOTES"
]

transicoes_A = [
    "T1_DOSAR_E_INSPEC",
    "T2_EJETAR_SILO701",
    "T3_REABASTECER_REC"
]

# Matriz Pre (|P| x |T|): fichas necessárias para disparar cada transição
Pre_A = [
    [1, 0, 0],  # P1_FN101_GRAOS
    [1, 0, 0],  # P2_AR_COMP_OK
    [1, 0, 0],  # P3_LINHA_LIVRE
    [0, 1, 0],  # P4_CLASSIFICANDO
    [0, 0, 0],  # P5_SILO701_LOTES
]

# Matriz Post (|P| x |T|): fichas produzidas após o disparo
Post_A = [
    [0, 0, 1],  # P1_FN101_GRAOS (T3 reabastece o funil)
    [0, 0, 1],  # P2_AR_COMP_OK (T3 recarrega o reservatório pneumático)
    [0, 1, 0],  # P3_LINHA_LIVRE (T2 conclui o lote e libera a linha)
    [1, 0, 0],  # P4_CLASSIFICANDO (T1 inicia transporte, visão e ejeção)
    [0, 1, 0],  # P5_SILO701_LOTES (T2 deposita lote classificado no SILO-701)
]

M0_A = (1, 1, 1, 0, 0)

rdp_ilimitada = RedeDePetri(lugares_A, transicoes_A, Pre_A, Post_A, M0_A)
rdp_ilimitada.exibir_matrizes("CENÁRIO A")

=== MATRIZ DE INCIDÊNCIA (C = Post - Pre) — CENÁRIO A ===
                  T1_DOSAR_E_INSPEC  T2_EJETAR_SILO701  T3_REABASTECER_REC
P1_FN101_GRAOS                   -1                  0                   1
P2_AR_COMP_OK                    -1                  0                   1
P3_LINHA_LIVRE                   -1                  1                   0
P4_CLASSIFICANDO                  1                 -1                   0
P5_SILO701_LOTES                  0                  1                   0


### 2. Implementação do Algoritmo de Karp-Miller (Árvore de Cobertura)
Quando uma Rede de Petri possui crescimento ilimitado de fichas em algum lugar $p_i$, a **Árvore de Atingibilidade** convencional torna-se infinita. O **Algoritmo de Karp-Miller** resolve esse problema introduzindo o símbolo $\omega$ (infinito dominável, onde $\omega + k = \omega$, $\omega - k = \omega$ e $\omega > n$ para todo $n \in \mathbb{N}$):

* Sempre que uma nova marcação $M'$ alcançada a partir de um ancestral $M$ no mesmo ramo satisfizer $M' \ge M$ componente a componente e $M' \neq M$, substituímos por $\omega$ todas as coordenadas $i$ onde $M'(p_i) > M(p_i)$.
* O algoritmo garante terminação finita para **qualquer** Rede de Petri.

In [2]:
OMEGA = float("inf")

def formatar_marcacao(M):
    return "(" + ", ".join("ω" if x == OMEGA else str(int(x)) for x in M) + ")"

def esta_habilitada(M, pre_col):
    """Verifica se M >= Pre(:, t) componente a componente."""
    return all(m_val >= req for m_val, req in zip(M, pre_col))

def disparar_transicao(M, pre_col, post_col):
    """Calcula M' = M - Pre(:, t) + Post(:, t) respeitando a aritmética com ω."""
    M_novo = []
    for m_val, req, prod in zip(M, pre_col, post_col):
        if m_val == OMEGA:
            M_novo.append(OMEGA)
        else:
            M_novo.append(int(m_val - req + prod))
    return tuple(M_novo)

def aplicar_aceleracao_omega(M_novo, caminho_ancestrais):
    """
    Regra de Karp-Miller: se existe ancestral M_anc no caminho tal que
    M_novo >= M_anc e M_novo != M_anc, substitui coordenadas estritamente maiores por ω.
    """
    M_acelerado = list(M_novo)
    for M_anc in caminho_ancestrais:
        if all(m_n >= m_a for m_n, m_a in zip(M_acelerado, M_anc)) and tuple(M_acelerado) != M_anc:
            for i in range(len(M_acelerado)):
                if M_acelerado[i] > M_anc[i]:
                    M_acelerado[i] = OMEGA
    return tuple(M_acelerado)

def construir_arvore_cobertura(rdp, max_nos=200):
    """Constrói a Árvore de Cobertura e o Grafo de Cobertura via busca em largura (BFS)."""
    arvore = nx.DiGraph()
    grafo_cobertura = nx.DiGraph()

    raiz_id = "S0"
    arvore.add_node(raiz_id, marcacao=rdp.M0, label=formatar_marcacao(rdp.M0), tipo="Raiz")
    grafo_cobertura.add_node(rdp.M0, label=formatar_marcacao(rdp.M0))

    marcacoes_exploradas = set()
    fila = deque([(raiz_id, rdp.M0, [rdp.M0])])
    contador_nos = 1

    while fila and contador_nos < max_nos:
        no_id, M_atual, ancestrais = fila.popleft()

        if M_atual in marcacoes_exploradas:
            arvore.nodes[no_id]["tipo"] = "Duplicado (Old)"
            continue

        marcacoes_exploradas.add(M_atual)
        habilitadas = []

        for j, t_nome in enumerate(rdp.transicoes):
            pre_col = rdp.Pre[:, j]
            post_col = rdp.Post[:, j]

            if esta_habilitada(M_atual, pre_col):
                habilitadas.append(t_nome)
                M_bruto = disparar_transicao(M_atual, pre_col, post_col)
                M_prox = aplicar_aceleracao_omega(M_bruto, ancestrais)

                filho_id = f"S{contador_nos}"
                contador_nos += 1

                tipo_no = "Duplicado (Old)" if M_prox in marcacoes_exploradas else "Ativo"
                arvore.add_node(filho_id, marcacao=M_prox, label=formatar_marcacao(M_prox), tipo=tipo_no)
                arvore.add_edge(no_id, filho_id, transicao=t_nome)

                grafo_cobertura.add_node(M_prox, label=formatar_marcacao(M_prox))
                grafo_cobertura.add_edge(M_atual, M_prox, transicao=t_nome)

                if M_prox not in marcacoes_exploradas:
                    fila.append((filho_id, M_prox, ancestrais + [M_prox]))

        if not habilitadas and arvore.nodes[no_id]["tipo"] != "Raiz":
            arvore.nodes[no_id]["tipo"] = "DEADLOCK (Terminal)"

    return arvore, grafo_cobertura

arvore_A, grafo_A = construir_arvore_cobertura(rdp_ilimitada)
print(f"Árvore de Cobertura (Cenário A) gerada com {arvore_A.number_of_nodes()} nós e {grafo_A.number_of_nodes()} estados distintos.")
for n, d in arvore_A.nodes(data=True):
    print(f"  {n:4s}: M = {d['label']:22s} | Status: {d['tipo']}")

Árvore de Cobertura (Cenário A) gerada com 14 nós e 7 estados distintos.
  S0  : M = (1, 1, 1, 0, 0)        | Status: Raiz
  S1  : M = (0, 0, 0, 1, 0)        | Status: Ativo
  S2  : M = (ω, ω, 1, 0, 0)        | Status: Ativo
  S3  : M = (0, 0, 1, 0, 1)        | Status: Ativo
  S4  : M = (ω, ω, 0, 1, 0)        | Status: Ativo
  S5  : M = (ω, ω, 0, 1, 0)        | Status: Duplicado (Old)
  S6  : M = (ω, ω, 1, 0, 0)        | Status: Duplicado (Old)
  S7  : M = (ω, ω, 1, 0, ω)        | Status: Ativo
  S8  : M = (ω, ω, 1, 0, ω)        | Status: Duplicado (Old)
  S9  : M = (ω, ω, 0, 1, 0)        | Status: Duplicado (Old)
  S10 : M = (ω, ω, 0, 1, ω)        | Status: Ativo
  S11 : M = (ω, ω, 1, 0, ω)        | Status: Duplicado (Old)
  S12 : M = (ω, ω, 1, 0, ω)        | Status: Duplicado (Old)
  S13 : M = (ω, ω, 0, 1, ω)        | Status: Duplicado (Old)


### 3. Diagnóstico de Segurança Operacional (Cenário A)
Observe nos estados gerados acima o surgimento do símbolo $\omega$ na 5ª posição (`P5_SILO701_LOTES`), bem como na recepção (`P1_FN101_GRAOS` e `P2_AR_COMP_OK`): **$( \omega, \omega, 1, 0, \omega )$**.

**Interpretação Física na Planta de Classificação de Grãos (Grupo 4):**
* O ciclo `T1_DOSAR_E_INSPEC` $\rightarrow$ `T2_EJETAR_SILO701` pode repetir-se indefinidamente enquanto `T3_REABASTECER_REC` alimenta novos lotes no funil, fazendo o volume de grãos acumulados no `SILO-701` crescer sem cota superior ($M(\text{P5}) \to \infty$).
* Em engenharia de automação, isso comprova formalmente o **risco de transbordamento (*overflow*) do `SILO-701` e do funil `FN-101`**, validando a obrigatoriedade dos intertravamentos de nível crítico `p_NC701` (`LIT-701`), `p_SILO_CHEIO` e `p_NC101` (`LIT-101`) projetados na Etapa 1.

### 4. Cenário B: Síntese de Controle via Lugares Complementares (Árvore de Atingibilidade Finita)
Para tornar a rede **limitada (*k-bounded*)** e eliminar o risco de transbordamento, adicionamos:
1. **`P6_SILO701_VAGA`:** Lugar complementar de espaço livre no `SILO-701` com capacidade física máxima $K_{\max} = 2$ lotes (equivalente à condição lógica $\neg p_{\text{NC701}}$ via `LIT-701`), impondo o Invariante-P:
   $$M(\text{P5\_SILO701\_LOTES}) + M(\text{P6\_SILO701\_VAGA}) = 2$$
2. **`P7_REC_VAGA`:** Lugar complementar de capacidade livre na recepção `FN-101` / reservatório ($K_{\max} = 1$, equivalente a $\neg p_{\text{NC101}}$ via `LIT-101`).
3. **`T4_DRENAR_AGV801`:** Transição de abertura da comporta `XV-701` e retirada de 1 lote classificado pelo veículo autoguiado `AGV-801` (integrando a Aula 22).

In [3]:
lugares_B = [
    "P1_FN101_GRAOS",
    "P2_AR_COMP_OK",
    "P3_LINHA_LIVRE",
    "P4_CLASSIFICANDO",
    "P5_SILO701_LOTES",
    "P6_SILO701_VAGA",   # Lugar complementar de capacidade do SILO-701 (K_max = 2)
    "P7_REC_VAGA"        # Lugar complementar de capacidade da recepção FN-101 (K_max = 1)
]

transicoes_B = [
    "T1_DOSAR_E_INSPEC",
    "T2_EJETAR_SILO701",
    "T3_REABASTECER_REC",
    "T4_DRENAR_AGV801"   # Abertura de XV-701 e expedição pelo AGV-801
]

Pre_B = [
    [1, 0, 0, 0],  # P1_FN101_GRAOS
    [1, 0, 0, 0],  # P2_AR_COMP_OK
    [1, 0, 0, 0],  # P3_LINHA_LIVRE
    [0, 1, 0, 0],  # P4_CLASSIFICANDO
    [0, 0, 0, 1],  # P5_SILO701_LOTES (T4 retira 1 lote para expedição)
    [0, 1, 0, 0],  # P6_SILO701_VAGA (T2 exige vaga livre no SILO-701!)
    [0, 0, 1, 0],  # P7_REC_VAGA (T3 exige vaga livre na recepção FN-101!)
]

Post_B = [
    [0, 0, 1, 0],  # P1_FN101_GRAOS
    [0, 0, 1, 0],  # P2_AR_COMP_OK
    [0, 1, 0, 0],  # P3_LINHA_LIVRE
    [1, 0, 0, 0],  # P4_CLASSIFICANDO
    [0, 1, 0, 0],  # P5_SILO701_LOTES
    [0, 0, 0, 1],  # P6_SILO701_VAGA (T4 libera vaga no SILO-701)
    [1, 0, 0, 0],  # P7_REC_VAGA (T1 libera vaga no funil FN-101)
]

# Marcação Inicial: 1 lote no FN-101, ar OK, linha livre, SILO-701 vazio (2 vagas), recepção cheia (0 vagas)
M0_B = (1, 1, 1, 0, 0, 2, 0)

rdp_limitada = RedeDePetri(lugares_B, transicoes_B, Pre_B, Post_B, M0_B)
arvore_B, grafo_B = construir_arvore_cobertura(rdp_limitada)

print(f"Árvore de Atingibilidade (Cenário B) gerada com {grafo_B.number_of_nodes()} estados alcançáveis finitos!")

Árvore de Atingibilidade (Cenário B) gerada com 12 estados alcançáveis finitos!


### 5. Auditoria Automática de Propriedades Comportamentais e Invariantes-P
A função abaixo inspeciona o **Grafo de Atingibilidade / Cobertura** resultante e verifica automaticamente todas as propriedades formais da Engenharia de Controle a Eventos Discretos (SED):
1. **Limitação ($k$-Boundedness):** Nenhum lugar ultrapassa $k$ fichas (ausência de $\omega$).
2. **Segurança (*Safeness* / 1-Boundedness):** Lugares críticos de exclusão mútua (`P3_LINHA_LIVRE` e `P4_CLASSIFICANDO`) nunca excedem $1$ ficha.
3. **Ausência de *Deadlocks*:** Todo estado alcançável possui pelo menos uma transição habilitada.
4. **Reversibilidade e Vivacidade (*L4-Liveness*):** O espaço de estados é fortemente conexo e todas as transições permanecem disparáveis.
5. **Verificação dos Invariantes-P:** Validação estrutural de conservação da linha ($P_3 + P_4 = 1$), do funil ($P_1 + P_7 = 1$) e do silo ($P_5 + P_6 = 2$).

In [4]:
def auditar_propriedades_rdp(rdp, grafo_estados, nome_cenario=""):
    estados = list(grafo_estados.nodes)
    n_lugares = len(rdp.lugares)

    max_por_lugar = [max(m[i] for m in estados) for i in range(n_lugares)]
    eh_limitada = all(v != OMEGA for v in max_por_lugar)
    eh_segura = all(v <= 1 for v in max_por_lugar)
    deadlocks = [m for m in estados if grafo_estados.out_degree(m) == 0]
    reversivel = nx.is_strongly_connected(grafo_estados) if eh_limitada else False
    transicoes_ativas = set(nx.get_edge_attributes(grafo_estados, "transicao").values())
    todas_vivas = (transicoes_ativas == set(rdp.transicoes)) and reversivel

    print("=" * 73)
    print(f"AUDITORIA FORMAL DE PROPRIEDADES — {nome_cenario}")
    print("=" * 73)
    print(f"• Total de Estados Distintos Alcançados: {len(estados)}")
    print(f"• Rede Limitada (k-Bounded):             {'SIM (Finita)' if eh_limitada else 'NÃO (Contém ω — Risco de Overflow!)'}")
    print(f"• Rede Segura (1-Bounded):               {'SIM' if eh_segura else 'NÃO (k > 1 em silos/recepção)'}")
    print(f"• Deadlocks Detectados:                  {len(deadlocks)} {deadlocks if deadlocks else '(Zero bloqueios)'}")
    print(f"• Sistema Reversível (Retorna a M0):     {'SIM' if reversivel else 'NÃO'}")
    print(f"• Vivacidade Global (L4-Live):           {'SIM' if todas_vivas else 'NÃO'}")
    print("\nCapacidade Máxima Observada por Lugar:")
    for lug, vmax in zip(rdp.lugares, max_por_lugar):
        val_str = "∞ (ω)" if vmax == OMEGA else str(vmax)
        print(f"  - {lug:20s}: max = {val_str}")
    print()
    return eh_limitada, len(deadlocks), reversivel, todas_vivas

auditar_propriedades_rdp(rdp_ilimitada, grafo_A, "CENÁRIO A (Sem Controle de Capacidade SILO-701)")
lim_B, dl_B, rev_B, viva_B = auditar_propriedades_rdp(
    rdp_limitada, grafo_B, "CENÁRIO B (Com Lugares Complementares e AGV-801)"
)

# Asserções formais de engenharia para o Cenário B
estados_B = list(grafo_B.nodes)
assert lim_B is True, "Erro: Cenário B deve ser estritamente limitado!"
assert dl_B == 0, "Erro: Cenário B não pode conter deadlocks!"
assert rev_B is True and viva_B is True, "Erro: Cenário B deve ser reversível e L4-Live!"
assert all(m[2] + m[3] == 1 for m in estados_B), "Violação do Invariante-P da Linha!"
assert all(m[4] + m[5] == 2 for m in estados_B), "Violação do Invariante-P do SILO-701!"
assert all(m[0] + m[6] == 1 for m in estados_B), "Violação do Invariante-P do FN-101!"

print(f"[OK] Invariantes-P validados em todos os {len(estados_B)} estados alcançáveis do Cenário B!")
print("     • Linha (P3 + P4 = 1):  VÁLIDO (Exclusão mútua garantida)")
print("     • Silo  (P5 + P6 = 2):  VÁLIDO (Zero risco de overflow em SILO-701)")
print("     • Funil (P1 + P7 = 1):  VÁLIDO (Zero risco de overflow em FN-101)")

AUDITORIA FORMAL DE PROPRIEDADES — CENÁRIO A (Sem Controle de Capacidade SILO-701)
• Total de Estados Distintos Alcançados: 7
• Rede Limitada (k-Bounded):             NÃO (Contém ω — Risco de Overflow!)
• Rede Segura (1-Bounded):               NÃO (k > 1 em silos/recepção)
• Deadlocks Detectados:                  0 (Zero bloqueios)
• Sistema Reversível (Retorna a M0):     NÃO
• Vivacidade Global (L4-Live):           NÃO

Capacidade Máxima Observada por Lugar:
  - P1_FN101_GRAOS      : max = ∞ (ω)
  - P2_AR_COMP_OK       : max = ∞ (ω)
  - P3_LINHA_LIVRE      : max = 1
  - P4_CLASSIFICANDO    : max = 1
  - P5_SILO701_LOTES    : max = ∞ (ω)

AUDITORIA FORMAL DE PROPRIEDADES — CENÁRIO B (Com Lugares Complementares e AGV-801)
• Total de Estados Distintos Alcançados: 12
• Rede Limitada (k-Bounded):             SIM (Finita)
• Rede Segura (1-Bounded):               NÃO (k > 1 em silos/recepção)
• Deadlocks Detectados:                  0 (Zero bloqueios)
• Sistema Reversível (Retorna a M0):    

### 6. Visualização Gráfica do Grafo de Atingibilidade Finito (Cenário B)
Plotamos o espaço de estados completo do controlador discreto corrigido da planta de classificação de grãos, destacando a marcação inicial $M_0$ em dourado e as transições de processo (`T1` a `T4`) que governam a operação segura e livre de *overflow*.

In [5]:
def plotar_grafo_atingibilidade(grafo, m0, titulo):
    fig, ax = plt.subplots(figsize=(14, 8))
    pos = nx.spring_layout(grafo, seed=42, k=1.4)

    cores = ["#FFD700" if n == m0 else "#5DADE2" for n in grafo.nodes]
    labels_nos = {n: d["label"] for n, d in grafo.nodes(data=True)}

    nx.draw_networkx_nodes(grafo, pos, node_color=cores, node_size=2200, edgecolors="black", linewidths=1.8, ax=ax)
    nx.draw_networkx_labels(grafo, pos, labels=labels_nos, font_size=8, font_weight="bold", ax=ax)

    nx.draw_networkx_edges(
        grafo, pos,
        edge_color="#34495E",
        width=1.6,
        arrows=True,
        arrowsize=18,
        connectionstyle="arc3,rad=0.12",
        ax=ax
    )

    edge_labels = {(u, v): d["transicao"].split("_")[0] for u, v, d in grafo.edges(data=True)}
    nx.draw_networkx_edge_labels(grafo, pos, edge_labels=edge_labels, font_size=8, font_color="#C0392B", ax=ax)

    ax.set_title(titulo, fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

plotar_grafo_atingibilidade(
    grafo_B,
    M0_B,
    "Grafo de Atingibilidade Finito — Classificação de Grãos e Silo Coletor SILO-701 (SCADA-Core Grupo 4)"
)